# Preference Tuning: RLHF & DPO

Companion notebook for the [RLHF & DPO lesson](https://ml-viz-ruby.vercel.app/courses/fine-tuning-alignment/04-rlhf-and-dpo).

**The idea in one sentence.** RLHF aligns a model to preferences by training a reward
model and optimising it with PPO under a KL leash — but **DPO** shows you can skip the
reward model and the RL entirely, optimising the *same* objective directly on the
preference pairs with a simple classification loss.

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\Big(\beta\big[(\log\pi_\theta(y_c)-\log\pi_{\text{ref}}(y_c)) - (\log\pi_\theta(y_r)-\log\pi_{\text{ref}}(y_r))\big]\Big)$$

The magic: this is Bradley–Terry on an *implicit* reward $\beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$,
so DPO raises the chosen response's probability relative to the reference while the
$\beta$ term keeps it from drifting too far.

We build DPO from scratch, **validate that it recovers the true preference ordering and
that $\beta$ controls the reference leash**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## Setup: a toy log-linear policy

Fix a small vocabulary `V = 6`. The policy is a single logits vector $\theta \in \mathbb{R}^V$ (no prompt conditioning — keeps the math readable). The 'prompt' is implicit.

$$
\pi_\theta(y) = \mathrm{softmax}(\theta)_y
$$

In [ ]:
V = 6
LABELS = [f'y{i}' for i in range(V)]

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def log_softmax(z):
    z = z - z.max()
    return z - np.log(np.exp(z).sum())

# A hidden 'true' reward — only used to *generate* the preference data.
# DPO never sees it directly.
true_reward = np.array([0.5, -0.2, 1.4, -0.6, 0.1, 0.9])
print('hidden true reward:', true_reward)

## Build a preference dataset

Sample a pair `(y_a, y_b)` uniformly, then label which one is 'chosen' with Bradley–Terry probability $\sigma(r(y_a) - r(y_b))$. This is the same generative model the lesson's RewardModel viz uses.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sample_pair(rng_local):
    y_a, y_b = rng_local.choice(V, size=2, replace=False)
    p_a_wins = sigmoid(true_reward[y_a] - true_reward[y_b])
    if rng_local.random() < p_a_wins:
        return int(y_a), int(y_b)   # chosen, rejected
    return int(y_b), int(y_a)

N_PAIRS = 4000
pairs = np.array([sample_pair(rng) for _ in range(N_PAIRS)])
chosen, rejected = pairs[:, 0], pairs[:, 1]
print('pairs shape:', pairs.shape)
print('first 5 pairs (chosen, rejected):', pairs[:5].tolist())

## Reference policy via SFT on the chosen responses

The RLHF / DPO setup assumes a competent reference policy $\pi_\text{ref}$ — usually the SFT checkpoint. We approximate it by maximum-likelihood training on the chosen labels only.

In [ ]:
theta_ref = np.zeros(V)
for _ in range(2000):
    idx = rng.integers(N_PAIRS)
    y = chosen[idx]
    p = softmax(theta_ref)
    grad = p.copy()
    grad[y] -= 1.0
    theta_ref -= 0.1 * grad

log_pi_ref = log_softmax(theta_ref)
print('reference policy probs:', np.round(softmax(theta_ref), 3))

## DPO training

For each pair $(y_c, y_r)$, compute the implicit reward margin

$$
m = \beta\big(\log \pi_\theta(y_c) - \log \pi_\text{ref}(y_c)\big) - \beta\big(\log \pi_\theta(y_r) - \log \pi_\text{ref}(y_r)\big)
$$

and loss $-\log \sigma(m)$. Gradient with respect to logits is $(\sigma(m) - 1) \cdot \beta \cdot (\nabla_\theta \log \pi_\theta(y_c) - \nabla_\theta \log \pi_\theta(y_r))$, where $\nabla_\theta \log \pi_\theta(y) = e_y - \pi_\theta$ for the softmax.

In [ ]:
def dpo_step(theta, y_c, y_r, log_pi_ref, beta=0.3, lr=0.5):
    log_pi = log_softmax(theta)
    pi = np.exp(log_pi)
    margin = beta * ((log_pi[y_c] - log_pi_ref[y_c]) - (log_pi[y_r] - log_pi_ref[y_r]))
    loss = -np.log(sigmoid(margin) + 1e-12)
    # grad wrt theta via the softmax identity above
    e_c = np.zeros_like(theta); e_c[y_c] = 1.0
    e_r = np.zeros_like(theta); e_r[y_r] = 1.0
    g_log_pi_c = e_c - pi
    g_log_pi_r = e_r - pi
    dloss_dmargin = sigmoid(margin) - 1.0
    grad = dloss_dmargin * beta * (g_log_pi_c - g_log_pi_r)
    return loss, theta - lr * grad

theta = theta_ref.copy()
losses, gaps = [], []
for step in range(3000):
    idx = rng.integers(N_PAIRS)
    loss, theta = dpo_step(theta, chosen[idx], rejected[idx], log_pi_ref)
    losses.append(loss)
    log_pi = log_softmax(theta)
    # average chosen-vs-rejected log-prob gap over the dataset
    gaps.append(float(np.mean(log_pi[chosen] - log_pi[rejected])))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
window = 100
axes[0].plot(np.convolve(losses, np.ones(window)/window, mode='valid'), color='#6366f1')
axes[0].set_xlabel('step'); axes[0].set_ylabel('DPO loss (smoothed)')
axes[0].set_title('DPO loss curve')
axes[1].plot(gaps, color='#22d3ee')
axes[1].axhline(0, ls='--', color='#94a3b8')
axes[1].set_xlabel('step'); axes[1].set_ylabel('mean log π(y_c) − log π(y_r)')
axes[1].set_title('Chosen-vs-rejected log-prob gap')
plt.tight_layout(); plt.show()

### Validate: $\beta=0$ removes the reference leash (loss floor $\log 2$)

The coefficient $\beta$ scales the implicit reward (the log-ratio to the reference). At
$\beta=0$ the margin is 0, so the loss is $-\log\sigma(0)=\log 2$ — the model gets no
preference signal. Larger $\beta$ means a stronger push but a tighter leash to
$\pi_{\text{ref}}$. We confirm the $\beta=0$ floor.

In [ ]:
def dpo_batch_loss(theta, y_c, y_r, log_pi_ref, beta):
    lp = log_softmax(theta)
    m = beta * ((lp[y_c] - log_pi_ref[y_c]) - (lp[y_r] - log_pi_ref[y_r]))
    return -np.log(sigmoid(m) + 1e-12)
th = rng.normal(size=V)
l0 = np.mean([dpo_batch_loss(th, chosen[i], rejected[i], log_pi_ref, 0.0) for i in range(200)])
lb = np.mean([dpo_batch_loss(th, chosen[i], rejected[i], log_pi_ref, 0.3) for i in range(200)])
print(f'DPO loss at beta=0.0: {l0:.4f}  (== log 2 = {np.log(2):.4f}, no signal)')
print(f'DPO loss at beta=0.3: {lb:.4f}')
assert abs(l0 - np.log(2)) < 1e-9, 'beta=0 collapses the DPO loss to log 2'
print('\n✅ beta scales the implicit reward / reference leash; beta=0 gives no preference signal')

## Sanity check: did DPO recover the preference ordering?

The trained policy should put more mass on responses with higher true reward — even though it never saw the true reward, only pairwise preferences.

In [ ]:
final = softmax(theta)
ref = softmax(theta_ref)
order = np.argsort(-true_reward)
x = np.arange(V)
w = 0.35
plt.bar(x - w/2, ref[order], w, label='π_ref (SFT)', color='#94a3b8')
plt.bar(x + w/2, final[order], w, label='π_θ (after DPO)', color='#f43f5e')
plt.xticks(x, [LABELS[i] for i in order])
plt.ylabel('probability')
plt.title('Policy probability, sorted by hidden true reward (high → low)')
plt.legend(); plt.show()

print('correlation(final policy, true reward):', np.round(np.corrcoef(final, true_reward)[0,1], 3))

### Validate: DPO recovers the hidden preference ordering

DPO never sees the hidden `true_reward` — only sampled preference pairs. Yet the
trained policy's probabilities should line up with the true reward (highest-reward
response gets the most mass). We assert a strong correlation and that the policy's
top choice matches the true best.

In [ ]:
corr = np.corrcoef(final, true_reward)[0, 1]
print(f'correlation(DPO policy, hidden true reward): {corr:.3f}')
print(f'DPO argmax response: y{int(np.argmax(final))}   true-best response: y{int(np.argmax(true_reward))}')
assert corr > 0.8, 'DPO should recover the hidden preference ordering'
assert np.argmax(final) == np.argmax(true_reward), 'DPO should place the most mass on the true-best response'
# and it moved AWAY from the (near-uniform) reference toward the preferred responses
assert final[np.argmax(true_reward)] > ref[np.argmax(true_reward)], 'DPO lifts the preferred response above the reference'
print('\n✅ DPO learned the preference ordering directly from pairs — no reward model, no RL')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **$\beta$ too small** | policy drifts far from the reference → degeneration/reward-hacking-like artifacts |
| **$\beta$ too large** | barely moves off the reference → weak alignment |
| **reference policy quality** | DPO is anchored to $\pi_{\text{ref}}$; a bad SFT reference caps results |
| **preference noise** | label noise sets the ceiling, same as reward-model RLHF |
| **DPO vs PPO** | DPO is simpler/stabler but PPO can use an online reward model and on-policy samples |

Demo: the implicit DPO reward $\beta\log(\pi_\theta/\pi_{\text{ref}})$ recovers the
hidden true reward.

In [ ]:
# DPO's implicit reward IS the log-ratio to the reference: r(y) = beta * log(pi_theta/pi_ref).
# We show this recovered reward correlates with the hidden true reward — DPO fit a reward
# model 'for free' inside the policy, which is why it needs no separate RM or PPO loop.
implicit_reward = 0.3 * (np.log(final + 1e-12) - np.log(ref + 1e-12))
c = np.corrcoef(implicit_reward, true_reward)[0, 1]
print('implicit DPO reward  beta*log(pi/pi_ref):', np.round(implicit_reward, 2))
print('hidden true reward                      :', np.round(true_reward, 2))
print(f'correlation: {c:.3f}')
assert c > 0.8, 'the implicit DPO reward should track the true reward'
print('\nDPO = Bradley-Terry on an implicit reward baked into the policy -> RLHF without the RL.')

## ✏️ Your turn

Implement `dpo_loss(theta, theta_ref_logits, chosen, rejected, beta)` that returns the **average** DPO loss across a batch of preference pairs. Verify it matches a known case (β = 0 → loss should be log 2 regardless of the policy).

In [ ]:
def dpo_loss(theta, theta_ref_logits, chosen, rejected, beta=0.3):
    # TODO(you): compute mean -log σ(β·(log π_θ(y_c) − log π_ref(y_c) − log π_θ(y_r) + log π_ref(y_r)))
    return 0.0

# β = 0 collapses the margin to 0 and the loss to -log σ(0) = log 2 ≈ 0.6931
l0 = dpo_loss(np.random.randn(V), np.random.randn(V), chosen[:200], rejected[:200], beta=0.0)
assert abs(l0 - np.log(2)) < 1e-6, f'β=0 should give log 2; got {l0}'
print('passed ✓  (loss at β=0 is', round(float(l0), 4), ')')

<details><summary>Solution</summary>

```python
def dpo_loss(theta, theta_ref_logits, chosen, rejected, beta=0.3):
    log_pi = log_softmax(theta)
    log_pi_ref = log_softmax(theta_ref_logits)
    m = beta * ((log_pi[chosen] - log_pi_ref[chosen])
                - (log_pi[rejected] - log_pi_ref[rejected]))
    return float(np.mean(-np.log(sigmoid(m) + 1e-12)))
```

Note: at β = 0 the margin is 0 for every pair, so the loss is $-\log \sigma(0) = \log 2$ regardless of the policy — a useful regression-test anchor.

</details>

## Key takeaways

- **DPO skips the reward model and PPO:** it optimises the same preference objective
  directly with a classification loss on (chosen, rejected) pairs.
- **It recovers the hidden ordering** (verified correlation with the unseen true
  reward) and places the most mass on the true-best response.
- **$\beta$ is the reference leash:** it scales the implicit reward
  $\beta\log\tfrac{\pi_\theta}{\pi_{\text{ref}}}$; $\beta=0$ gives no signal (loss
  $\log 2$), large $\beta$ pushes harder but risks drifting from $\pi_{\text{ref}}$.
- **The reference policy anchors alignment** — DPO raises chosen probabilities
  *relative to* the SFT reference, not in absolute terms.